# Part 4 : K-Means Clustering

**Durée estimée : 2h30**

## Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** la différence entre apprentissage supervisé et non-supervisé
2. **Décrire** l'algorithme K-Means étape par étape
3. **Maîtriser** les hyperparamètres de K-Means
4. **Évaluer** la qualité des clusters (silhouette, inertie, coude)
5. **Interpréter** les clusters dans un contexte business

---

## Problème Réel : Comment Spotify crée-t-il vos playlists personnalisées ?

Chaque vendredi, Spotify vous propose une playlist "Discover Weekly" avec 30 chansons que vous n'avez jamais écoutées... mais que vous allez probablement adorer.

**Question :** Comment Spotify peut-il deviner vos goûts musicaux sans que vous ayez explicitement dit "j'aime le rock alternatif des années 90" ?

*(Prenez 30 secondes pour réfléchir...)*

<details>
<summary>Votre intuition ?</summary>

### Réponse

Spotify ne vous demande pas vos genres préférés. Il **observe** votre comportement et vous **groupe** avec des utilisateurs similaires.

C'est du **clustering** : regrouper des éléments similaires **sans avoir de catégories prédéfinies**.

</details>

### La segmentation client : un cas d'étude réel

Selon une [étude 2024 (MDPI)](https://www.mdpi.com/2813-2203/2/4/42) sur 541 909 clients UK, le clustering K-Means a identifié **trois segments distincts** avec des stratégies marketing différentes.

| Segment | Caractéristiques | Stratégie marketing |
|---------|------------------|---------------------|
| High-Value | Dépenses élevées, fréquent | Programme VIP |
| At-Risk | Était actif, plus maintenant | Réactivation |
| New | Premier achat récent | Onboarding |

---

## 4.1 Intuition : Supervisé vs Non-Supervisé

```
┌───────────────────────────────────────────────────────────────────────┐
│           SUPERVISÉ vs NON-SUPERVISÉ                                  │
├───────────────────────────────────────────────────────────────────────┤
│                                                                       │
│  SUPERVISÉ (Parts 1-3)              NON-SUPERVISÉ (Part 4)            │
│  ─────────────────────              ───────────────────────           │
│                                                                       │
│  "Voici des exemples avec           "Voici des données.              │
│   leurs réponses. Apprends."         Trouve des groupes."            │
│                                                                       │
│  Données:                           Données:                          │
│  ┌──────┬────────┐                  ┌──────┐                          │
│  │ X    │ y      │                  │ X    │  (pas de y !)            │
│  ├──────┼────────┤                  ├──────┤                          │
│  │ 🏠   │ 250k€  │                  │ 👤   │                          │
│  │ 🏠   │ 180k€  │                  │ 👤   │                          │
│  └──────┴────────┘                  └──────┘                          │
│                                                                       │
│  Exemples:                          Exemples:                         │
│  • Prédire le prix (régression)     • Segmenter les clients          │
│  • Détecter spam (classification)   • Regrouper des documents        │
│  • Approuver un prêt               • Détecter des anomalies          │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

**Question Socratique :** Pourquoi ne fait-on pas de train/test split en clustering ?

*(Réponse : Pas de "vraie réponse" à vérifier ! On veut segmenter TOUS les clients, pas en cacher 20%.)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
# Créer un dataset de clients e-commerce
np.random.seed(42)

# 3 groupes de clients
g1 = pd.DataFrame({'age': np.random.normal(25, 5, 100),
                   'revenu_annuel': np.random.normal(25000, 5000, 100),
                   'score_depense': np.random.normal(70, 10, 100)})
g2 = pd.DataFrame({'age': np.random.normal(45, 10, 100),
                   'revenu_annuel': np.random.normal(75000, 15000, 100),
                   'score_depense': np.random.normal(50, 15, 100)})
g3 = pd.DataFrame({'age': np.random.normal(60, 8, 100),
                   'revenu_annuel': np.random.normal(45000, 10000, 100),
                   'score_depense': np.random.normal(25, 10, 100)})

clients = pd.concat([g1, g2, g3], ignore_index=True)
clients['age'] = clients['age'].clip(18, 80)
clients['revenu_annuel'] = clients['revenu_annuel'].clip(15000, 150000)
clients['score_depense'] = clients['score_depense'].clip(1, 100)

print("Aperçu des données clients (SANS labels !) :")
clients.head()

In [ ]:
# Visualiser : voyez-vous des groupes ?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(clients['revenu_annuel'], clients['score_depense'], alpha=0.6)
axes[0].set_xlabel('Revenu annuel (€)')
axes[0].set_ylabel('Score de dépense')
axes[0].set_title('Voyez-vous des groupes ?')

axes[1].scatter(clients['age'], clients['score_depense'], alpha=0.6)
axes[1].set_xlabel('Âge')
axes[1].set_ylabel('Score de dépense')
axes[1].set_title('Âge vs Dépense')

plt.tight_layout()
plt.show()

---

## 4.2 Construction : L'Algorithme K-Means

```
┌───────────────────────────────────────────────────────────────────────┐
│              ALGORITHME K-MEANS (pour K=3 clusters)                   │
├───────────────────────────────────────────────────────────────────────┤
│                                                                       │
│  ÉTAPE 1: Choisir K centres au hasard (centroïdes)                    │
│        ·  ·     ·                                                     │
│     ·    ★        ·     ★ = centroïde                                │
│   ·   ·    ★   ·                                                      │
│      ·  ·    ·   ★                                                    │
│                                                                       │
│  ÉTAPE 2: Assigner chaque point au centre le plus proche              │
│        🔴  🔴     🔵                                                   │
│     🔴    ★        🔵     Chaque point prend la couleur               │
│   🔴   🔴    ★   🔵       de son centre le plus proche                │
│      🟢  🟢    🟢   ★                                                  │
│                                                                       │
│  ÉTAPE 3: Recalculer les centres (moyenne de chaque groupe)           │
│        🔴  🔴     🔵                                                   │
│     🔴  ★          🔵     ★ se déplace vers le centre                 │
│   🔴   🔴      ★  🔵       de son groupe                              │
│      🟢  🟢  ★  🟢                                                     │
│                                                                       │
│  ÉTAPE 4: Répéter étapes 2-3 jusqu'à convergence                      │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Normaliser les données (CRITIQUE pour K-Means !)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clients)

# Pourquoi ? K-Means utilise la distance.
# Sans normalisation, le revenu (milliers) dominerait l'âge (dizaines)

In [ ]:
# Appliquer K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clients['cluster'] = kmeans.fit_predict(X_scaled)

print("Clusters assignés !")
clients.head()

In [ ]:
# Visualiser les clusters
colors = ['red', 'blue', 'green']
plt.figure(figsize=(10, 6))

for i in range(3):
    mask = clients['cluster'] == i
    plt.scatter(clients.loc[mask, 'revenu_annuel'], 
                clients.loc[mask, 'score_depense'],
                c=colors[i], label=f'Cluster {i}', alpha=0.6)

plt.xlabel('Revenu annuel (€)')
plt.ylabel('Score de dépense')
plt.title('Clients segmentés par K-Means')
plt.legend()
plt.show()

---

## 4.3 Hyperparamètres : Contrôler K-Means

```
┌───────────────────────────────────────────────────────────────────────┐
│     HYPERPARAMÈTRES DE KMeans                                        │
├───────────────────────────────────────────────────────────────────────┤
│                                                                       │
│  n_clusters = 3                                                       │
│  └─ Nombre de clusters à trouver (K)                                  │
│     • Le PLUS IMPORTANT : détermine le résultat                       │
│     • Utiliser la méthode du coude pour le choisir                    │
│                                                                       │
│  init = 'k-means++'                                                   │
│  └─ Méthode d'initialisation des centroïdes                           │
│     • 'k-means++' (défaut) : intelligent, évite mauvais départs       │
│     • 'random' : aléatoire, moins stable                              │
│                                                                       │
│  n_init = 10                                                          │
│  └─ Nombre de fois qu'on relance l'algo avec différents départs       │
│     • Plus = meilleur résultat, mais plus lent                        │
│     • Sklearn garde le meilleur des n_init essais                     │
│                                                                       │
│  max_iter = 300                                                       │
│  └─ Nombre max d'itérations par essai                                 │
│     • Augmenter si l'algo ne converge pas                             │
│                                                                       │
│  random_state = 42                                                    │
│  └─ Graine aléatoire pour reproductibilité                            │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Impact de n_init sur la stabilité
print("═" * 55)
print("IMPACT DE n_init (nombre d'initialisations)")
print("═" * 55)

for n_init in [1, 5, 10, 20]:
    inerties = []
    for _ in range(5):
        km = KMeans(n_clusters=3, n_init=n_init, random_state=None)
        km.fit(X_scaled)
        inerties.append(km.inertia_)
    print(f"n_init={n_init:2d} : Inertie moyenne={np.mean(inerties):.1f} ± {np.std(inerties):.1f}")

print("\n→ Plus n_init est grand, plus les résultats sont stables")

In [ ]:
# Comparaison init='k-means++' vs init='random'
print("═" * 55)
print("COMPARAISON DES MÉTHODES D'INITIALISATION")
print("═" * 55)

for init_method in ['k-means++', 'random']:
    km = KMeans(n_clusters=3, init=init_method, n_init=10, random_state=42)
    km.fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_)
    print(f"init='{init_method:12s}' : Inertie={km.inertia_:.1f}, Silhouette={sil:.3f}")

print("\n→ 'k-means++' donne généralement de meilleurs résultats")

### Résumé des hyperparamètres

| Hyperparamètre | Défaut | Quand changer ? |
|----------------|--------|------------------|
| n_clusters | 8 | TOUJOURS ! Utiliser méthode du coude |
| init | 'k-means++' | Rarement (déjà optimal) |
| n_init | 10 | Augmenter si résultats instables |
| max_iter | 300 | Augmenter si warning convergence |

---

## 4.4 Évaluation : Mesurer la Qualité des Clusters

En clustering, pas de "vraie réponse". Mais on peut mesurer :
- **L'inertie** : cohésion interne (plus bas = mieux)
- **Le score silhouette** : séparation entre clusters (plus haut = mieux)

### La Méthode du Coude (Elbow Method)

**Problème** : Comment choisir K sans connaître le "bon" nombre de groupes ?

**Solution** : Tracer l'inertie pour différents K et chercher le "coude".

In [ ]:
# Méthode du coude
inerties = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coude (Inertie)
axes[0].plot(K_range, inerties, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='Coude (K=3)')
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du Coude')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='Optimal (K=3)')
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Score Silhouette')
axes[1].set_title('Score Silhouette (plus haut = mieux)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMeilleur K selon silhouette : {list(K_range)[silhouettes.index(max(silhouettes))]}")

### Interpréter le Score Silhouette

```
┌─────────────────────────────────────────────────────────────────────────┐
│ SCORE DE SILHOUETTE                                                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│ • +1 = Point très bien dans son cluster, loin des autres               │
│ •  0 = Point à la frontière entre deux clusters                        │
│ • -1 = Point probablement mal assigné                                  │
│                                                                         │
│ Interprétation de la moyenne :                                          │
│ • > 0.7  : Excellente structure                                         │
│ • 0.5-0.7 : Bonne structure                                             │
│ • 0.25-0.5 : Structure faible                                           │
│ • < 0.25 : Pas de structure claire                                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Évaluation finale de notre clustering K=3
print("═" * 55)
print("ÉVALUATION DU CLUSTERING (K=3)")
print("═" * 55)
print(f"\nInertie         : {kmeans.inertia_:.2f}")
print(f"Score Silhouette : {silhouette_score(X_scaled, kmeans.labels_):.3f}")

# Profil des clusters
print("\n" + "═" * 55)
print("PROFIL DE CHAQUE CLUSTER")
print("═" * 55)

profil = clients.groupby('cluster').agg({
    'age': 'mean',
    'revenu_annuel': 'mean',
    'score_depense': 'mean'
}).round(1)
profil['nb_clients'] = clients.groupby('cluster').size()
print(profil)

In [ ]:
# Interprétation business
print("\n" + "═" * 55)
print("INTERPRÉTATION BUSINESS")
print("═" * 55)
print("""
🎯 Cluster 0 : Seniors économes
   - Âge élevé, revenu moyen, faible dépense
   → Offres promotionnelles, réductions

🎯 Cluster 1 : Jeunes dépensiers  
   - Jeunes, faible revenu, forte dépense
   → Paiement fractionné, offres tendance

🎯 Cluster 2 : Familles aisées
   - Âge moyen, revenu élevé, dépense moyenne
   → Produits premium, programme fidélité
""")

---

## Exercice Pratique : Segmenter des clients bancaires

Analysez les comportements de cartes de crédit.

In [ ]:
# Dataset cartes de crédit
np.random.seed(123)

credit_cards = pd.DataFrame({
    'solde_moyen': np.concatenate([
        np.random.uniform(100, 1000, 200),
        np.random.uniform(2000, 8000, 200),
        np.random.uniform(500, 2000, 100)
    ]),
    'nb_achats_mois': np.concatenate([
        np.random.uniform(1, 10, 200),
        np.random.uniform(20, 50, 200),
        np.random.uniform(10, 25, 100)
    ]),
    'limite_credit': np.concatenate([
        np.random.uniform(2000, 5000, 200),
        np.random.uniform(10000, 30000, 200),
        np.random.uniform(5000, 15000, 100)
    ])
})

print("Dataset Cartes de Crédit :")
credit_cards.head()

### Mission :
1. Normaliser avec StandardScaler
2. Méthode du coude pour trouver K
3. Appliquer K-Means
4. Calculer le score silhouette
5. Interpréter les clusters

In [ ]:
# À VOUS DE JOUER !

# 1. Normaliser
# scaler_cc = ...

# 2. Coude
# ...

# 3. K-Means
# ...

# 4. Silhouette
# ...

# 5. Interpréter
# ...

In [ ]:
# SOLUTION

# 1. Normaliser
scaler_cc = StandardScaler()
X_cc_scaled = scaler_cc.fit_transform(credit_cards)

# 2. Coude
inerties_cc = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cc_scaled)
    inerties_cc.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 8), inerties_cc, 'bo-')
plt.xlabel('K')
plt.ylabel('Inertie')
plt.title('Méthode du Coude - Cartes de Crédit')
plt.show()

# 3. K-Means (K=3 d'après le coude)
kmeans_cc = KMeans(n_clusters=3, random_state=42, n_init=10)
credit_cards['cluster'] = kmeans_cc.fit_predict(X_cc_scaled)

# 4. Silhouette
sil_cc = silhouette_score(X_cc_scaled, kmeans_cc.labels_)
print(f"\nScore Silhouette : {sil_cc:.3f}")

# 5. Interpréter
print("\n" + "═" * 55)
print("PROFILS DES SEGMENTS")
print("═" * 55)
print(credit_cards.groupby('cluster').mean().round(1))

print("""
🏷️ Cluster 0 : "Utilisateurs occasionnels"
   → Cible pour upgrade de carte

🏷️ Cluster 1 : "Power Users"
   → Programme fidélité premium

🏷️ Cluster 2 : "Utilisateurs modérés"
   → Potentiel de croissance
""")

---

## Récapitulatif

### Structure de cette partie :

| Section | Contenu |
|---------|--------|
| **Hook** | Spotify playlists, segmentation client |
| **4.1 Intuition** | Supervisé vs Non-supervisé |
| **4.2 Construction** | Algorithme K-Means (assigner, recalculer, répéter) |
| **4.3 Hyperparamètres** | n_clusters, init, n_init, max_iter |
| **4.4 Évaluation** | Méthode du coude, score silhouette |

### Code essentiel :

```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 1. Normaliser (OBLIGATOIRE !)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Trouver K (méthode du coude)
inerties = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inerties.append(km.inertia_)

# 3. Appliquer K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# 4. Évaluer
sil = silhouette_score(X_scaled, clusters)
```

### Prochaine partie : Quand Utiliser Quoi ?

Maintenant que vous connaissez 4 algorithmes, comment choisir le bon ?

---

## Réflexion Métacognitive

1. Pourquoi la normalisation est-elle OBLIGATOIRE pour K-Means ?
2. Comment interpréter un score silhouette de 0.45 ?
3. Dans votre domaine, voyez-vous une application du clustering ?

---

**Sources :**
- [MDPI 2024 - Clustering Algorithms for Customer Segmentation](https://www.mdpi.com/2813-2203/2/4/42)
- [scikit-learn KMeans Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)